# Model B v2 — Unfrozen Backbone (controlled comparison with Model A)

Purpose: give Model B the same training power as Model A. Original Model B froze MobileNetV2 entirely; Model A fine-tunes it fully. This retrains Model B's temporal architecture with the backbone **unfrozen**, matching Model A's training regime (lr=1e-4, weight_decay=1e-4, same optimizer applied to all parameters).

Reuses the existing temporal crops from Weeks 3-4 (`train_temporal_crops.h5` etc, still the un-corrected sampling window, that fix is a separate step). This isolates the backbone-freezing question specifically.

**Batch size reduced 16 → 8**: backpropagating through all 16 frames per sample, with the backbone now trainable, needs much more GPU memory than the frozen version did. This is a memory necessity, not a methodology change.

**Result (test set): F1 = 0.766, precision = 0.757, recall = 0.776** — unfreezing the backbone made Model B's test F1 WORSE than the frozen version (0.787), likely due to overfitting: only 876 training samples against 2.9M trainable parameters. Train F1 reached 0.99 by epoch 6-7 while val plateaued then declined. This suggests the original frozen-backbone design was the more appropriate choice for this dataset size, not a methodological shortcut.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/PIE_thesis'
CROPS_DIR = f'{DRIVE_ROOT}/crops'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
CHECKPOINTS_DIR = f'{DRIVE_ROOT}/checkpoints'

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
import h5py
import random
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
import pandas as pd

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', device)


## 2. Dataset: reuses existing temporal crops (same files Model B v1 used)

In [ ]:
IMG_SIZE = 224
normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class PIETemporalDataset(Dataset):
    def __init__(self, h5_path, train=False):
        self.h5_path = h5_path
        self.train = train
        with h5py.File(h5_path, 'r') as hf:
            self.length = hf['images'].shape[0]
            self.t_frames = hf['images'].shape[1]
        self._hf = None

    def _ensure_open(self):
        if self._hf is None:
            self._hf = h5py.File(self.h5_path, 'r')

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        self._ensure_open()
        frames = self._hf['images'][idx]
        label = int(self._hf['labels'][idx])
        flip = self.train and random.random() < 0.5
        out = torch.zeros(self.t_frames, 3, IMG_SIZE, IMG_SIZE)
        for t in range(self.t_frames):
            img = torch.from_numpy(frames[t]).permute(2, 0, 1).float() / 255.0
            if flip:
                img = torch.flip(img, dims=[2])
            out[t] = normalize(img)
        return out, label

train_ds = PIETemporalDataset(f'{CROPS_DIR}/train_temporal_crops.h5', train=True)
val_ds = PIETemporalDataset(f'{CROPS_DIR}/val_temporal_crops.h5')
test_ds = PIETemporalDataset(f'{CROPS_DIR}/test_temporal_crops.h5')
print('Dataset sizes:', len(train_ds), len(val_ds), len(test_ds))
print('T frames:', train_ds.t_frames)

BATCH_SIZE = 8  # reduced from 16: backbone is now trainable, needs more memory
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 3. Class weights (same as before)

In [ ]:
with h5py.File(f'{CROPS_DIR}/train_temporal_crops.h5', 'r') as hf:
    train_labels = hf['labels'][:]

unique, counts = np.unique(train_labels, return_counts=True)
print('Train class distribution:', dict(zip(unique.tolist(), counts.tolist())))
class_weights = torch.tensor([len(train_labels) / c for c in counts], dtype=torch.float32).to(device)
print('Class weights:', class_weights)


## 4. Model: MobileNetV2 backbone (now UNFROZEN) + LSTM + classifier

The backbone is now a trainable submodule of the model itself, not a separate no-grad feature extractor. Gradients flow all the way back through it during training, matching how Model A trains.

In [ ]:
FEATURE_DIM = 1280
HIDDEN_DIM = 128

class TemporalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).features
        self.lstm = nn.LSTM(input_size=FEATURE_DIM, hidden_size=HIDDEN_DIM, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.Linear(HIDDEN_DIM, 2)

    def forward(self, frames_batch):
        b, t, c, h, w = frames_batch.shape
        flat = frames_batch.view(b * t, c, h, w)
        feats = self.backbone(flat)
        feats = nn.functional.adaptive_avg_pool2d(feats, 1).view(b * t, -1)
        feats = feats.view(b, t, -1)
        _, (h_n, _) = self.lstm(feats)
        out = self.dropout(h_n[-1])
        return self.classifier(out)

model = TemporalModel().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_trainable:,}')


## 5. Train (same regularization pattern: best checkpoint + early stopping)

In [ ]:
EPOCHS = 20
PATIENCE = 5

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train):
        for frames, y in loader:
            frames, y = frames.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            out = model(frames)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * frames.size(0)
            n += frames.size(0)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return total_loss / max(n, 1), f1

history = []
best_val_f1 = 0.0
patience_counter = 0
for epoch in range(EPOCHS):
    train_loss, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_f1 = run_epoch(val_loader, train=False)
    print(f'Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f} train_f1={train_f1:.4f}  val_loss={val_loss:.4f} val_f1={val_f1:.4f}')
    history.append({'epoch': epoch+1, 'train_loss': train_loss, 'train_f1': train_f1, 'val_loss': val_loss, 'val_f1': val_f1})
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), f'{CHECKPOINTS_DIR}/model_b_v2_unfrozen.pt')
        print(f'  New best val_f1={val_f1:.4f}, checkpoint saved')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}, best val_f1={best_val_f1:.4f}')
            break

print('Training done, best val_f1:', best_val_f1)


## 6. Evaluate on held-out test set

In [ ]:
model.load_state_dict(torch.load(f'{CHECKPOINTS_DIR}/model_b_v2_unfrozen.pt'))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for frames, y in test_loader:
        frames = frames.to(device)
        out = model(frames)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

f1 = f1_score(all_labels, all_preds, zero_division=0)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)

results = {'model': 'Model B v2 - Temporal LSTM, Unfrozen Backbone', 'f1': f1, 'precision': precision, 'recall': recall}
print(results)
print(classification_report(all_labels, all_preds, target_names=['not-crossing', 'crossing']))

metrics_df = pd.DataFrame([results])
metrics_df.to_csv(f'{RESULTS_DIR}/model_b_v2_test_metrics.csv', index=False)
history_df = pd.DataFrame(history)
history_df.to_csv(f'{RESULTS_DIR}/model_b_v2_training_history.csv', index=False)


## 7. What this tells us

Compare against:
- Model A (unfrozen backbone): F1 = 0.795
- Model B v1 (frozen backbone): F1 = 0.787

With both models now trained the same way (backbone unfrozen, same optimizer settings), this is the fairest comparison available before the sampling-window fix. Whatever gap remains here is attributable to single-frame vs. temporal architecture, not training regime.

Note: this still uses the uncorrected sampling window (frames spread across the whole track, not the last 2.0s). The sampling fix and re-extraction remain a separate, still-pending step.

**Actual result: F1 = 0.766.** Lower than both Model A and Model B v1, see note at top of notebook on why (overfitting from backbone capacity vs. dataset size).